In [1]:
# Imports
import torch
import torchvision
from torchvision import transforms, datasets
import torch.nn as nn

# Optional: mount Google Drive (if not already mounted)
from google.colab import drive
drive.mount('/content/drive')

# Directories for training and validation data
train_dir = '/content/drive/MyDrive/processed_data/train'
val_dir = '/content/drive/MyDrive/processed_data/test'

# Data augmentation and normalization for training and validation
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet normalization:contentReference[oaicite:0]{index=0}
])
val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load datasets (expects directory structure root/class/filename):contentReference[oaicite:1]{index=1}
train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
val_dataset = datasets.ImageFolder(val_dir, transform=val_transforms)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader   = torch.utils.data.DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=4)

# Setup device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load EfficientNet-B0 with pretrained ImageNet weights:contentReference[oaicite:2]{index=2}
model = torchvision.models.efficientnet_b0(weights=torchvision.models.EfficientNet_B0_Weights.IMAGENET1K_V1)

# Modify the final classifier: add dropout and output layer for our classes
num_classes = len(train_dataset.classes)  # e.g., 2 for PD vs HC
num_ftrs = model.classifier[1].in_features  # get input features of original final layer:contentReference[oaicite:3]{index=3}
model.classifier = nn.Sequential(
    nn.Dropout(p=0.5, inplace=True),
    nn.Linear(num_ftrs, num_classes)
)  # Dropout regularization before final classifier:contentReference[oaicite:4]{index=4}
model = model.to(device)

# Define loss function and optimizer (Adam with weight decay for L2 regularization)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

# Training loop with early stopping
best_val_acc = 0.0
epochs_no_improve = 0
patience = 3
best_model_path = '/content/drive/MyDrive/Model/efficientnet_b0_model.pth'

for epoch in range(20):
    model.train()
    # Training phase
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    # Validation phase
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    val_acc = correct / total
    print(f'Epoch {epoch+1}, Validation Accuracy: {val_acc:.4f}')
    # Check for improvement
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)  # save the best model
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
    # Early stopping condition: stop if no improvement for 'patience' epochs:contentReference[oaicite:5]{index=5}
    if epochs_no_improve >= patience:
        print("Early stopping triggered")
        break


Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 107MB/s] 


Epoch 1, Validation Accuracy: 0.9000
Epoch 2, Validation Accuracy: 0.9425
Epoch 3, Validation Accuracy: 0.9500
Epoch 4, Validation Accuracy: 0.9700
Epoch 5, Validation Accuracy: 0.9675
Epoch 6, Validation Accuracy: 0.9725
Epoch 7, Validation Accuracy: 0.9800
Epoch 8, Validation Accuracy: 0.9800
Epoch 9, Validation Accuracy: 0.9850
Epoch 10, Validation Accuracy: 0.9875
Epoch 11, Validation Accuracy: 0.9925
Epoch 12, Validation Accuracy: 0.9775
Epoch 13, Validation Accuracy: 0.9900
Epoch 14, Validation Accuracy: 0.9900
Early stopping triggered
